# Tugas Praktikum 6: Apache Spark

**Mata Kuliah:** Praktikum Big Data
**Nama:** Arkan Ubaidillah Warman
**NIM:** 2411537001
**Program Studi:** Informatika, Fakultas Teknologi Informasi, Universitas Andalas

Tugas memakai dataset transaksi dari bagian E.2. Notebook ini berjalan di runtime tersendiri, jadi SparkSession, dataset, dan kolom `total` dibuat ulang di bagian persiapan.

## Persiapan

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, round

spark = SparkSession.builder \
    .appName("TugasPraktikum6") \
    .master("local[*]") \
    .getOrCreate()

print("Versi Spark:", spark.version)

Versi Spark: 4.3.0


In [3]:
data_transaksi = [
    (1, "Laptop", "Elektronik", 7500000, 2),
    (2, "Mouse", "Aksesoris", 150000, 5),
    (3, "Keyboard", "Aksesoris", 300000, 3),
    (4, "Monitor", "Elektronik", 2500000, 2),
    (5, "Flashdisk", "Aksesoris", 80000, 10),
    (6, "Printer", "Elektronik", 1800000, 1),
    (7, "Headset", "Aksesoris", 250000, 4),
    (8, "Laptop", "Elektronik", 7500000, 1),
    (9, "Mouse", "Aksesoris", 150000, 8),
    (10, "Monitor", "Elektronik", 2500000, 3)
]

kolom = ["id_transaksi", "produk", "kategori", "harga", "jumlah"]

df = spark.createDataFrame(data_transaksi, kolom)

# Kolom total = harga x jumlah (sama seperti E.2 Langkah 7)
df_total = df.withColumn("total", col("harga") * col("jumlah"))
df_total.show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
+------------+---------+----------+-------+------+--------+



## Tugas 1. Analisis Produk

Lima produk dengan total pendapatan tertinggi, memakai DataFrame API. Satu produk bisa muncul di beberapa transaksi (Laptop, Mouse, dan Monitor masing-masing dua kali), jadi pendapatannya dijumlahkan per produk dulu dengan `groupBy("produk")`, lalu diurutkan menurun dan diambil lima teratas dengan `limit(5)`.

In [4]:
top5_produk = df_total.groupBy("produk") \
    .agg(
        sum("total").alias("total_pendapatan"),
        sum("jumlah").alias("unit_terjual")
    ) \
    .orderBy(col("total_pendapatan").desc()) \
    .limit(5)

top5_produk.show()

+-------+----------------+------------+
| produk|total_pendapatan|unit_terjual|
+-------+----------------+------------+
| Laptop|        22500000|           3|
|Monitor|        12500000|           5|
|  Mouse|         1950000|          13|
|Printer|         1800000|           1|
|Headset|         1000000|           4|
+-------+----------------+------------+



Laptop memimpin dengan Rp22.500.000 dari 3 unit, disusul Monitor Rp12.500.000 dari 5 unit. Mouse berada di posisi ketiga dengan Rp1.950.000 walaupun harganya murah, karena terjual 13 unit. Keyboard dan Flashdisk tidak masuk lima besar.

## Tugas 2. Analisis Kategori

Total pendapatan, jumlah transaksi, dan rata-rata jumlah produk terjual untuk setiap kategori dihitung dalam satu `agg()`.

In [5]:
analisis_kategori = df_total.groupBy("kategori") \
    .agg(
        sum("total").alias("total_pendapatan"),
        count("*").alias("jumlah_transaksi"),
        round(avg("jumlah"), 2).alias("rata_rata_jumlah_produk")
    ) \
    .orderBy(col("total_pendapatan").desc())

analisis_kategori.show()

+----------+----------------+----------------+-----------------------+
|  kategori|total_pendapatan|jumlah_transaksi|rata_rata_jumlah_produk|
+----------+----------------+----------------+-----------------------+
|Elektronik|        36800000|               5|                    1.8|
| Aksesoris|         4650000|               5|                    6.0|
+----------+----------------+----------------+-----------------------+



Elektronik menghasilkan Rp36.800.000 dari 5 transaksi, sedangkan Aksesoris Rp4.650.000 dari 5 transaksi. Polanya terbalik pada jumlah unit: pembeli Aksesoris rata-rata membeli 6 unit per transaksi, pembeli Elektronik hanya 1,8 unit. Elektronik unggul karena harga satuannya tinggi, bukan karena volumenya.

## Tugas 3. Filtering Data

Transaksi dengan harga produk lebih dari Rp100.000 **dan** jumlah minimal 3 unit. Kedua kondisi digabung dengan operator `&`, dan setiap kondisi diberi tanda kurung.

In [6]:
df_filter_tugas = df_total.filter(
    (col("harga") > 100000) & (col("jumlah") >= 3)
)

df_filter_tugas.show()
print("Jumlah transaksi yang memenuhi:", df_filter_tugas.count())

+------------+--------+----------+-------+------+-------+
|id_transaksi|  produk|  kategori|  harga|jumlah|  total|
+------------+--------+----------+-------+------+-------+
|           2|   Mouse| Aksesoris| 150000|     5| 750000|
|           3|Keyboard| Aksesoris| 300000|     3| 900000|
|           7| Headset| Aksesoris| 250000|     4|1000000|
|           9|   Mouse| Aksesoris| 150000|     8|1200000|
|          10| Monitor|Elektronik|2500000|     3|7500000|
+------------+--------+----------+-------+------+-------+



Jumlah transaksi yang memenuhi: 5


Ada 5 transaksi yang memenuhi: Mouse (5 unit), Keyboard (3 unit), Headset (4 unit), Mouse (8 unit), dan Monitor (3 unit). Flashdisk terbeli 10 unit tetapi tidak lolos karena harganya Rp80.000, di bawah batas Rp100.000.

## Tugas 4. Spark SQL

Nama produk dan total pendapatan setiap produk, diurutkan dari pendapatan terbesar.

In [7]:
df_total.createOrReplaceTempView("transaksi")

spark.sql("""
    SELECT produk,
           SUM(total) AS total_pendapatan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_pendapatan DESC
""").show()

+---------+----------------+
|   produk|total_pendapatan|
+---------+----------------+
|   Laptop|        22500000|
|  Monitor|        12500000|
|    Mouse|         1950000|
|  Printer|         1800000|
|  Headset|         1000000|
| Keyboard|          900000|
|Flashdisk|          800000|
+---------+----------------+



Lima baris teratas query SQL sama dengan hasil Tugas 1 yang memakai DataFrame API. Query ini menampilkan ketujuh produk, termasuk Keyboard (Rp900.000) dan Flashdisk (Rp800.000) di urutan terbawah.

## Tugas 5. Analisis Konsep: RDD dan DataFrame

RDD adalah kumpulan objek terdistribusi tanpa skema. Spark tidak tahu isi setiap elemen, sehingga pengolahan ditulis sendiri lewat fungsi seperti `map()` dan `filter()` dengan lambda, dan Spark tidak bisa mengoptimalkannya. DataFrame menyusun data dalam baris dan kolom bernama dengan tipe yang jelas, seperti terlihat pada `printSchema()`. Karena skemanya diketahui, Catalyst optimizer bisa menyusun rencana eksekusi yang efisien, dan kita bisa memakai operasi tingkat tinggi seperti `groupBy()`, `agg()`, atau SQL.

Contoh kondisi penggunaan RDD: data tidak terstruktur seperti baris log mentah atau teks bebas yang perlu dipecah dengan logika khusus, misalnya menghitung kata dengan `flatMap()` lalu `reduceByKey()`. Contoh kondisi penggunaan DataFrame: data tabel seperti transaksi penjualan pada praktikum ini, ketika kita butuh agregasi per kategori, filter dengan beberapa kondisi, join antar-tabel, atau query SQL.

Sel berikut memperlihatkan perbedaannya pada tugas yang sama: total pendapatan per kategori dengan RDD dan dengan DataFrame.

In [8]:
# RDD: struktur data ditangani sendiri lewat indeks tuple
rdd_transaksi = spark.sparkContext.parallelize(data_transaksi)
rdd_kategori = rdd_transaksi \
    .map(lambda t: (t[2], t[3] * t[4])) \
    .reduceByKey(lambda a, b: a + b)
print("RDD       :", sorted(rdd_kategori.collect()))

# DataFrame: cukup sebut nama kolom
df_kategori = df_total.groupBy("kategori").agg(sum("total").alias("total_pendapatan"))
print("DataFrame :", sorted([tuple(r) for r in df_kategori.collect()]))

RDD       : [('Aksesoris', 4650000), ('Elektronik', 36800000)]


DataFrame : [('Aksesoris', 4650000), ('Elektronik', 36800000)]


Kedua cara memberi angka yang sama. Versi RDD harus tahu bahwa kategori ada di indeks 2, harga di indeks 3, dan jumlah di indeks 4. Versi DataFrame cukup menyebut nama kolom, jadi lebih mudah dibaca dan dioptimalkan oleh Spark.